In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import glob
import numpy as np
import math
import gc
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
import timm
import random

In [ ]:
# Set seeds for reproducibility
RANDOM_STATE = 42

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


# Data directory (update this path)
data_dir = "/content/drive/MyDrive/Dataset/castor_v2_224x224"

# Device configuration
print(f"Using device: {device}")

Using device: cuda:0


In [ ]:
# ============================================================================
# Dataset
# ============================================================================

class PestDataset(Dataset):
    """Dataset for pest classification - images already resized to 224x224"""
    def __init__(self, image_paths, labels, transform=None, classes=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.classes = classes  # List of class names

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


def get_transforms():
    """Get train and validation transforms - NO resizing since images are already 224x224"""
    # ImageNet normalization
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    return train_transform, val_transform

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        'total': total_params,
        'trainable': trainable_params
    }



# ============================================================================
# Data Loading
# ============================================================================

class OptimizedDataLoader:
    def __init__(self, data_dir, batch_size, num_classes=3, n_splits=5, fold_index=0):
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.n_splits = n_splits
        self.fold_index = fold_index
        self.classes = None
        self.cls2idx = None
        self.train_size = 0
        self.val_size = 0
        self.dataset = None  # Will be set after loading

    def load_dataset(self):
        # Load image paths
        image_paths = []
        for ext in ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.png', '*.PNG']:
            image_paths.extend(glob.glob(os.path.join(self.data_dir, '*', ext)))

        print(f"Total number of images found: {len(image_paths)}")

        # Extract labels
        labels = [os.path.basename(os.path.dirname(p)) for p in image_paths]
        self.classes = sorted(list(set(labels)))
        self.cls2idx = {c: i for i, c in enumerate(self.classes)}
        int_labels = np.array([self.cls2idx[c] for c in labels])
        image_paths = np.array(image_paths)

        # Stratified K-Fold split
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=RANDOM_STATE)
        all_folds_indices = list(skf.split(image_paths, int_labels))

        if self.fold_index < 0 or self.fold_index >= self.n_splits:
            raise ValueError(f"fold_index {self.fold_index} is out of range [0, {self.n_splits-1}]")

        train_indices, val_indices = all_folds_indices[self.fold_index]

        train_paths = image_paths[train_indices]
        train_labels = int_labels[train_indices]
        val_paths = image_paths[val_indices]
        val_labels = int_labels[val_indices]

        self.train_size = len(train_paths)
        self.val_size = len(val_paths)

        print(f"\n TRAINING DATA ANALYSIS (Fold {self.fold_index + 1}/{self.n_splits}):")
        print(f" Training images: {self.train_size}")
        print(f" Validation images: {self.val_size}")
        print(f" Batch size: {self.batch_size}")
        print(f" Steps per epoch: {math.ceil(self.train_size / self.batch_size)}")

        # Print class distribution
        self._print_class_distribution(train_labels, "Training")
        self._print_class_distribution(val_labels, "Validation")

        # Get transforms
        train_transform, val_transform = get_transforms()

        # Create datasets
        train_dataset = PestDataset(train_paths, train_labels, transform=train_transform, classes=self.classes)
        val_dataset = PestDataset(val_paths, val_labels, transform=val_transform, classes=self.classes)

        # Store dataset reference for accessing classes
        self.dataset = train_dataset

        # Create dataloaders
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=2,
            pin_memory=True,
            drop_last=False
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        return train_loader, val_loader

    def _print_class_distribution(self, labels, dataset_name):
        counts = Counter(labels)
        total = len(labels)
        print(f"\n CLASS DISTRIBUTION ({dataset_name}):")
        for cls_idx, count in sorted(counts.items()):
            cls_name = self.classes[cls_idx] if self.classes else str(cls_idx)
            percentage = (count / total) * 100
            print(f"    {cls_name}: {count:3d} images ({percentage:5.1f}%)")

    def get_dataloaders(self):
        return self.load_dataset()

In [ ]:
# ============================================================================
# Training Functions
# ============================================================================

# ----------------------- Plot Confusion Matrix -----------------------
def plot_cm(cm, class_names, title):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)

    img_path = f"{title.replace(' ', '_')}.png"
    plt.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close()
    return img_path


def train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        probs = F.softmax(outputs, dim=-1)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })

    if scheduler is not None:
        scheduler.step()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    # Calculate metrics for training
    train_prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    train_rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    train_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    try:
        train_auc = roc_auc_score(all_labels, all_probs, average='macro', multi_class='ovr')
    except:
        train_auc = 0.0

    train_metrics = {
        'precision': train_prec,
        'recall': train_rec,
        'f1': train_f1,
        'roc_auc': train_auc
    }

    return epoch_loss, epoch_acc, train_metrics, all_labels, all_preds, all_probs

In [ ]:
def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validating"):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = F.softmax(outputs, dim=-1)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    val_loss = running_loss / len(all_labels)
    val_acc = accuracy_score(all_labels, all_preds)
    val_prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    val_rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    try:
        val_auc = roc_auc_score(all_labels, all_probs, average='macro', multi_class='ovr')
    except:
        val_auc = 0.0

    metrics = {
        'loss': val_loss,
        'accuracy': val_acc,
        'precision': val_prec,
        'recall': val_rec,
        'f1_score': val_f1,
        'roc_auc': val_auc
    }

    return metrics, all_labels, all_preds, all_probs

In [ ]:
def train_with_kfold(data_dir, n_splits=5, epochs=100, project_name="MobileViT_KFold", group_name="Original"):
    """Training with K-Fold cross validation"""
    results = []
    all_fold_data = []  # Store best metrics for final summary table

    config = {
        'batch_size': 16,
        'num_classes': 3,
        'learning_rate': 1e-4,
        'weight_decay': 0.1,
        'usewandb':False
    }

    for fold in range(n_splits):
        print(f"\n{'='*60}")
        print(f"FOLD {fold+1}/{n_splits}")
        print(f"{'='*60}")

        is_last_fold = (fold == n_splits - 1)

        # Wandb config for this fold
        fold_config = {
            'fold': fold + 1,
            'batch_size': config['batch_size'],
            'epochs': epochs,
            'learning_rate': config['learning_rate'],
            'weight_decay': config['weight_decay'],
            'dropout': 0.5,
            'model': 'MobileViT-XXS',
            'patience': 20,
            'patch_size': 4,
            'drop_path_rate': 0.1,
            'fusion': 'elementwise_addition',
            'num_mobilevit_blocks': 2
        }

        # Initialize W&B for this fold
        run_name = f"{group_name}_Fold-{fold + 1}"
        if config.get("usewandb"):
            wandb.init(
                project=project_name,
                name=run_name,
                group=group_name,
                config=fold_config,
                reinit=True
            )

        # Create data loader
        loader = OptimizedDataLoader(
            data_dir=data_dir,
            batch_size=config['batch_size'],
            num_classes=config['num_classes'],
            n_splits=n_splits,
            fold_index=fold
        )

        train_loader, val_loader = loader.get_dataloaders()

        # Build model
        model = timm.create_model('mobilevit_xxs', pretrained=True, num_classes=config['num_classes'])
        model = model.to(device)

        if fold == 0:
            # Print model summary
            total_params = sum(p.numel() for p in model.parameters())
            trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"\nTotal parameters: {total_params:,}")
            print(f"Trainable parameters: {trainable_params:,}")

        # Loss function with label smoothing
        criterion = nn.CrossEntropyLoss()

        # Optimizer
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay'],
            betas=(0.9, 0.999),
            eps=1e-8
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=3, factor=0.3
    )

        # Training loop
        best_val_acc = 0.0
        best_val_loss = float('inf')
        best_epoch = -1
        best_prec = best_rec = best_f1 = best_roc = 0
        best_val_cm = None
        best_train_cm = None
        best_val_report = None
        best_train_report = None
        patience = 20
        patience_counter = 0
        best_model_path = f'MobileViT_{group_name}_fold{fold+1}.pth'

        # Get class names from loader
        class_names = loader.dataset.classes

        for epoch in range(epochs):
            # Train
            train_loss, train_acc, train_metrics, train_labels, train_preds, train_probs = train_one_epoch(
                model, train_loader, criterion, optimizer, scheduler, device, epoch
            )

            # Validate
            val_metrics, val_labels, val_preds, val_probs = validate(model, val_loader, criterion, device)

            print(f"\nEpoch {epoch+1}/{epochs}")
            print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
            print(f"  Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.4f}")
            print(f"  Val Precision: {val_metrics['precision']:.4f}, Val Recall: {val_metrics['recall']:.4f}")
            print(f"  Val F1: {val_metrics['f1_score']:.4f}, Val AUC: {val_metrics['roc_auc']:.4f}")

            # Get current learning rate
            current_lr = optimizer.param_groups[0]['lr']

            # Log to W&B
            if config.get("usewandb"):
                wandb.log({
                    'epoch': epoch + 1,
                    'learning_rate': current_lr,
                    'train_loss': train_loss,
                    'train_accuracy': train_acc,
                    'train_precision': train_metrics['precision'],
                    'train_recall': train_metrics['recall'],
                    'train_f1': train_metrics['f1'],
                    'train_auc': train_metrics['roc_auc'],
                    'val_loss': val_metrics['loss'],
                    'val_accuracy': val_metrics['accuracy'],
                    'val_precision': val_metrics['precision'],
                    'val_recall': val_metrics['recall'],
                    'val_f1': val_metrics['f1_score'],
                    'val_auc': val_metrics['roc_auc']
                })

            # Save best model
            if val_metrics['accuracy'] > best_val_acc:
                best_val_acc = val_metrics['accuracy']
                best_val_loss = val_metrics['loss']
                best_prec = val_metrics['precision']
                best_rec = val_metrics['recall']
                best_f1 = val_metrics['f1_score']
                best_roc = val_metrics['roc_auc']
                best_epoch = epoch + 1
                patience_counter = 0

                # Save model (state_dict only)
                torch.save(model.state_dict(), best_model_path)
                if config.get("usewandb"):
                    wandb.save(best_model_path)

                print(f"  ✓ New best model saved! Val Acc: {best_val_acc:.4f}")

                # Store confusion matrices and classification reports
                best_val_cm = confusion_matrix(val_labels, val_preds)
                best_val_report = classification_report(val_labels, val_preds, target_names=class_names, output_dict=False)
                best_train_cm = confusion_matrix(train_labels, train_preds)
                best_train_report = classification_report(train_labels, train_preds, target_names=class_names, output_dict=False)
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"\nEarly stopping triggered after {epoch+1} epochs")
                    break

        # Log confusion matrices and classification reports
        if best_val_cm is not None:
            val_cm_img = plot_cm(best_val_cm, class_names, f"Fold{fold+1}_Validation_CM")
            train_cm_img = plot_cm(best_train_cm, class_names, f"Fold{fold+1}_Training_CM")
            if config.get("usewandb"):
                wandb.log({
                    "best_validation_confusion_matrix": wandb.Image(val_cm_img),
                    "best_training_confusion_matrix": wandb.Image(train_cm_img),
                    "best_validation_classification_report": wandb.Html(f"<pre>{best_val_report}</pre>"),
                    "best_training_classification_report": wandb.Html(f"<pre>{best_train_report}</pre>")
                })

        # Store this fold's metrics in summary
        if config.get("usewandb"):
            wandb.summary["fold"] = fold + 1
            wandb.summary["best_epoch"] = best_epoch
            wandb.summary["best_val_loss"] = best_val_loss
            wandb.summary["final_val_loss"] = best_val_loss
            wandb.summary["final_val_accuracy"] = best_val_acc
            wandb.summary["final_precision"] = best_prec
            wandb.summary["final_recall"] = best_rec
            wandb.summary["final_f1"] = best_f1
            wandb.summary["final_roc_auc"] = best_roc

        # Log summary table on last fold
        if is_last_fold:
            # Add current fold's data
            all_fold_data.append([
                fold + 1, best_epoch, best_val_acc, best_val_loss, best_prec, best_rec, best_f1, best_roc
            ])

            # Create summary table with ALL folds
            if config.get("usewandb"):
                summary_table = wandb.Table(
                    columns=["fold", "best_epoch", "best_accuracy", "best_loss", "best_precision", "best_recall", "best_f1", "best_roc_auc"]
                )
                for fold_data in all_fold_data:
                    summary_table.add_data(*fold_data)
                wandb.log({"all_folds_best_metrics": summary_table})

        # Load best model for final evaluation
        model.load_state_dict(torch.load(best_model_path))
        final_metrics, final_labels, final_preds, final_probs = validate(model, val_loader, criterion, device)

        results.append({
            'fold': fold + 1,
            'accuracy': final_metrics['accuracy'],
            'precision': final_metrics['precision'],
            'recall': final_metrics['recall'],
            'f1_score': final_metrics['f1_score'],
            'roc_auc': final_metrics['roc_auc'],
            'best_val_accuracy': best_val_acc,
            'best_val_loss': best_val_loss
        })

        # Store data for folds 1-4 (fold 5 adds itself inside the is_last_fold block)
        if not is_last_fold:
            all_fold_data.append([
                fold + 1, best_epoch, best_val_acc, best_val_loss, best_prec, best_rec, best_f1, best_roc
            ])

        print(f"\nFold {fold+1} Final Results:")
        print(f"  Accuracy: {final_metrics['accuracy']:.4f}")
        print(f"  Precision: {final_metrics['precision']:.4f}")
        print(f"  Recall: {final_metrics['recall']:.4f}")
        print(f"  F1-Score: {final_metrics['f1_score']:.4f}")
        print(f"  ROC-AUC: {final_metrics['roc_auc']:.4f}")

        print(f"{'='*60}")
        if config.get("usewandb"):
            wandb.finish()

        # Cleanup
        del model, optimizer, scheduler, train_loader, val_loader
        torch.cuda.empty_cache()
        gc.collect()

    # Summary
    print(f"\n{'='*60}")
    print("OVERALL K-FOLD CROSS VALIDATION RESULTS")
    print(f"{'='*60}")

    for result in results:
        print(f"✓ Fold {result['fold']}: Acc={result['accuracy']:.4f}, "
              f"Prec={result['precision']:.4f}, Rec={result['recall']:.4f}, "
              f"F1={result['f1_score']:.4f}, AUC={result['roc_auc']:.4f}")

    mean_acc = np.mean([r['accuracy'] for r in results])
    std_acc = np.std([r['accuracy'] for r in results])
    mean_prec = np.mean([r['precision'] for r in results])
    std_prec = np.std([r['precision'] for r in results])
    mean_rec = np.mean([r['recall'] for r in results])
    std_rec = np.std([r['recall'] for r in results])
    mean_f1 = np.mean([r['f1_score'] for r in results])
    std_f1 = np.std([r['f1_score'] for r in results])
    mean_auc = np.mean([r['roc_auc'] for r in results])
    std_auc = np.std([r['roc_auc'] for r in results])

    print(f"\nMean Results ({n_splits} folds):")
    print(f"  Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"  Precision: {mean_prec:.4f} ± {std_prec:.4f}")
    print(f"  Recall: {mean_rec:.4f} ± {std_rec:.4f}")
    print(f"  F1-Score: {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"  ROC-AUC: {mean_auc:.4f} ± {std_auc:.4f}")

    return results

In [ ]:
# ============================================================================
# Main
# ============================================================================

if __name__ == "__main__":
    print("="*70)
    print("MobileViT-XXS Pretrained Fine-Tuning")
    print("="*70)

    # Test model creation
    print("\n1️⃣  Creating pretrained MobileViT-XXS model...")
    model = timm.create_model('mobilevit_xxs', pretrained=True, num_classes=3)

    # Test forward pass
    x = torch.randn(2, 3, 224, 224)
    with torch.no_grad():
        y = model(x)
    print(f"   Input shape: {x.shape}")
    print(f"   Output shape: {y.shape}")

    # Print model summary
    params_before = count_parameters(model)
    print(f"\n   📊 Model Parameters:")
    print(f"      Total: {params_before['total']:,}")
    print(f"      Trainable: {params_before['trainable']:,}")

    print("\n2️⃣  Architecture Summary:")
    print("   ✓ Using pretrained MobileViT-XXS from timm")
    print("   ✓ Fine-tuning entire model")
    print("   ✓ Loss function: CrossEntropyLoss")

    print("\n✅ Model ready for training!")
    print("="*70)

    # Run training with wandb logging
    print("\n🚀 Starting training with W&B logging...")
    results = train_with_kfold(
        data_dir=data_dir,
        n_splits=5,
        epochs=100,
        project_name="MobileViT_KFold_ImageNet",
        group_name="Original"
    )

    print("\n✅ Training complete! Check W&B for detailed results.")


Using device: cuda
MobileViT-XXS Pretrained Fine-Tuning

1️⃣  Creating pretrained MobileViT-XXS model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/5.14M [00:00<?, ?B/s]

   Input shape: torch.Size([2, 3, 224, 224])
   Output shape: torch.Size([2, 3])

   📊 Model Parameters (before pruning):
      Total: 951,987
      Trainable: 951,987

2️⃣  Applying 20% structured pruning (L1-norm)...
✂️  Applied 20% structured pruning to 35 Conv2d layers
   Pruned layers: ['stem.conv', 'stages.0.0.conv1_1x1.conv', 'stages.0.0.conv2_kxk.conv', 'stages.0.0.conv3_1x1.conv', 'stages.1.0.conv1_1x1.conv']...

   📊 Model Parameters (after pruning):
      Total: 951,987
      Non-zero: 951,984
      Pruned: 3 (0.0%)

3️⃣  Architecture Changes Summary:
   ✓ Using pretrained MobileViT-XXS from timm
   ✓ Fine-tuning entire model
   ✓ Loss function: CrossEntropyLoss
   ✓ Structured pruning: 20% L1-norm filter pruning

   💾 Model size reduction after pruning: ~0%

✅ Model ready for training!

🚀 Starting training with W&B logging...

FOLD 1/5


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...


Epoch 42/100
  Train Loss: 0.0089, Train Acc: 0.9975
  Val Loss: 0.1028, Val Acc: 0.9798
  Val Precision: 0.9804, Val Recall: 0.9649
  Val F1: 0.9714, Val AUC: 0.9993

Early stopping triggered after 42 epochs


Validating: 100%|██████████| 13/13 [00:01<00:00, 12.90it/s]



Fold 5 Final Results:
  Accuracy: 0.9899
  Precision: 0.9833
  Recall: 0.9896
  F1-Score: 0.9862
  ROC-AUC: 0.9993


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
learning_rate,██████████▇▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▁
train_accuracy,▁▅▆▇▇▇▇█████████████████████████████████
train_auc,▁▆▇▇████████████████████████████████████
train_f1,▁▅▆▇▇▇▇█████████████████████████████████
train_loss,█▆▄▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_precision,▁▅▆▇▇▇▇██▇██████████████████████████████
train_recall,▁▅▆▇▇▇▇▇███▇████████████████████████████
val_accuracy,▁▆▇▅▆▆▇▅▇▇▇▇▇▆▇███▇▇█▇▇▇▇▇▆▇▆▇▇▇▇▇▇█▇█▇▇
val_auc,▁▇█▇██████████████████████████▇█████████
+4,...



OVERALL K-FOLD CROSS VALIDATION RESULTS
✓ Fold 1: Acc=0.9799, Prec=0.9758, Rec=0.9692, F1=0.9722, AUC=0.9972
✓ Fold 2: Acc=0.9848, Prec=0.9851, Rec=0.9744, F1=0.9790, AUC=0.9947
✓ Fold 3: Acc=0.9798, Prec=0.9804, Rec=0.9658, F1=0.9719, AUC=0.9967
✓ Fold 4: Acc=0.9899, Prec=0.9899, Rec=0.9825, F1=0.9859, AUC=0.9989
✓ Fold 5: Acc=0.9899, Prec=0.9833, Rec=0.9896, F1=0.9862, AUC=0.9993

Mean Results (5 folds):
  Accuracy: 0.9849 ± 0.0045
  Precision: 0.9829 ± 0.0047
  Recall: 0.9763 ± 0.0087
  F1-Score: 0.9790 ± 0.0063
  ROC-AUC: 0.9974 ± 0.0016

✅ Training complete! Check W&B for detailed results.
